# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [1]:
%pip install evaluate rouge_score

  Obtaining dependency information for evaluate from https://files.pythonhosted.org/packages/3e/af/3e990d8d4002bbc9342adb4facd59506e653da93b2417de0fa6027cb86b1/evaluate-0.4.6-py3-none-any.whl.metadata
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Obtaining dependency information for absl-py from https://files.pythonhosted.org/packages/18/a6/907a406bb7d359e6a63f99c313846d9eec4f7e6f7437809e03aa00fa3074/absl_py-2.4.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=0c70845e56ba6d6f1c26dd21717291b71c04932d5ce673708af5617cca2da75f
  Stored in directory: /Users/natalija.glisovic/Library/Caches/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully 

We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [2]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [3]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

/Users/natalija.glisovic/Documents/phd_course_work/Deep Learning for NLP/labs/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 54948/54948 [00:00<00:00, 332483.54 examples/s]


In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [4]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter: 100%|██████████| 54948/54948 [00:01<00:00, 48836.28 examples/s]


In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [ ]:
def format_input_output(example):
  # `messages` is a list of messages, each with a `content` string and a `role`.
  messages = example['messages'] #get list of message dics 

  prompt_parts = [] #will hold formatted messages for the prompt
  response = "" #will hold the assistant's response

  for msg in messages:
      role = msg['role'] #assistant, user, system
      content = msg['content'] #actual text of the message
      if role == 'assistant': #what model should generate/append 
          response = content + "<|im_end|>"
      else:
          prompt_parts.append(f"<|im_start|>{role}\n{content}<|im_end|>\n") #other messages go into the prompt

  prompt = "".join(prompt_parts) + "<|im_start|>assistant\n" #join all prompt parts and add the start of the assistant's response

  return {"prompt": prompt, "response": response}

Comment: We want model to learn to generate response, not memorize input, so we split the conversation at assistant bounday so prompt (system + user turns) passed as contexts and its tokens masked in label so cross-entropy loss ignores. ChatML tags like im_start and im_end tell the model where each speaker begins and ends, which matches what its pretrained on. so the im_end marks stopping poin.

Apply the function you implemented to the dataset as a whole.

In [8]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map: 100%|██████████| 400/400 [00:00<00:00, 9047.69 examples/s]


Then verify that the dataset now contains the new fields you created.

In [9]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more food after the restaurant ran out.<|im_end|>'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [13]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"] #tokenize prompt without adding special tokens (we already have them in the text)
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"] #tokenize response without adding special tokens (we already have them in the text)

    input_ids = prompt_ids + response_ids #concatenate prompt and response token ids to create the input for the model
    labels = [-100] * len(prompt_ids) + response_ids #labels are the same as input_ids but with prompt token ids replaced by -100 so that loss is only calculated on response tokens
    attention_mask = [1] * len(input_ids) #attention mask is 1 for all tokens since we want the model to attend to all of them

    return {
        "input_ids": input_ids,       # Input token ids of the prompt and response
        "attention_mask": attention_mask,  # Attention mask of the prompt and response
        "labels": labels,          # Output token ids of the prompt (masked) and response
    }


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [14]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)

Map: 100%|██████████| 400/400 [00:00<00:00, 3135.01 examples/s]



## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [15]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [16]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [17]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [21]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json 
print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=False, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 16432.14it/s]
/Users/natalija.glisovic/Documents/phd_course_work/Deep Learning for NLP/labs/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 400/400 [01:05<00:00,  6.15it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.6193437576293945,
  "eval_model_preparation_time": 0.0023,
  "eval_rougeL": 0.5741986905444035,
  "eval_runtime": 65.2118,
  "eval_samples_per_second": 6.134,
  "eval_steps_per_second": 6.134,
  "epoch": 0
}


Comment: Why ROUGE-L score high even without training for instruction following? So ROUGE-L scores looks at longest subsequence of tokens between model reference and ouput so longest chain of words that appear in same order, so only looks at length of reference not meaning. E.g. "dog sat by the door" and "the cat sat near the door" then. we have "the sat the door" so 4 words. 

So even without instruction tuning, the model will produce e.g. a, the, you (common words) so we will have overlap and thus high score and also due to pre-training data overlap as our SmolLM2 was trained on large web data so it probabvly includes text similar to SmolTalk responses so it can generate plausible sounding continuation that "inflates" score 


## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [22]:

baseline_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=False, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

baseline_trainer = make_trainer(base_model, baseline_training_args) #what performs SFT -computes cross-entory on response token and updayes all weights via backprop.

baseline_trainer.train() # Train the model for 1 epoch on the training set
baseline_eval_metrics = baseline_trainer.evaluate() # Evaluate the model on the test set
print(baseline_eval_metrics) # Print the evaluation metrics for the fine-tuned model

  0%|          | 0/5000 [00:00<?, ?it/s]/Users/natalija.glisovic/Documents/phd_course_work/Deep Learning for NLP/labs/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 40%|████      | 2000/5000 [10:16<14:06,  3.55it/s]

{'loss': '1.684', 'grad_norm': '7.562', 'learning_rate': '3.001e-05', 'epoch': '0.4'}


 80%|████████  | 4000/5000 [21:18<05:40,  2.94it/s]

{'loss': '1.483', 'grad_norm': '10.56', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


                                                   
100%|██████████| 5000/5000 [28:18<00:00,  2.94it/s]


{'eval_loss': '1.521', 'eval_rougeL': '0.6301', 'eval_runtime': '84.75', 'eval_samples_per_second': '4.72', 'eval_steps_per_second': '4.72', 'epoch': '1'}
{'train_runtime': '1699', 'train_samples_per_second': '2.943', 'train_steps_per_second': '2.943', 'train_loss': '1.548', 'epoch': '1'}


100%|██████████| 400/400 [01:26<00:00,  4.64it/s]

{'eval_loss': 1.5205596685409546, 'eval_rougeL': 0.6301315435201437, 'eval_runtime': 86.4252, 'eval_samples_per_second': 4.628, 'eval_steps_per_second': 4.628, 'epoch': 1.0}


Comment: Loss drops here so our SFT model assigns higher probability to correct response token, so byyer at predicting the kind of outputs dataset expects. ROUGE-L improved as well so SFT model generates responses with more token overlap so beter at following instruction-response format and more relevant conten. ROUGE-L improvvement (0.574 to 0.630) not massive improvement but I only trained model for one epoch and 5k examples so not fully converged yet, but still improvments.

### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [25]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """

    return sum(p.numel() for p in model.parameters() if p.requires_grad) #pnumel for total number of elements in the parameter tensor, sum over all trainable parameters to get total number of trainable parameters (requires_grad=True means it's a trainable parameter)
    raise NotImplementedError()

Apply this function to the SFT-trained model and check that the result makes sense.

In [26]:
print(num_trainable_parameters(base_model))

134515008


In [27]:
#verify that the number of trainable parameters is the same as the total number of parameters in the model, which means all parameters are being updated during training (no freezing) and that we are not using parameter efficient fine-tuning methods like LoRA or adapters.
total = sum(p.numel() for p in base_model.parameters())
trainable = num_trainable_parameters(base_model)
print(f"Trainable: {trainable:,} / Total: {total:,}")

Trainable: 134,515,008 / Total: 134,515,008


## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [28]:
def extract_lora_targets(model):
    targets = {}
    for name, module in model.named_modules():
        if name.endswith(("q_proj", "v_proj")):
            targets[name] = module
    return targets

We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [29]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [ ]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        self.W = W
        for param in self.W.parameters():
            param.requires_grad = False  # freeze original weights

        in_features = W.in_features
        out_features = W.out_features
        self.scaling = alpha / r

        self.A = nn.Linear(in_features, r, bias=False)   # down-projection, meaning it takes input of size in_features and projects it down to size r
        self.B = nn.Linear(r, out_features, bias=False)  # up-projection, meaning it takes input of size r and projects it back up to size out_features
        nn.init.zeros_(self.B.weight)  # B=0 at init so ΔW=BA=0 at start

    def forward(self, x):
        return self.W(x) + self.scaling * self.B(self.A(x)) #forward pass computes original output W(x) plus low-rank update (α/r) * B(A(x)) where A projects input down to r dimensions, B projects it back up to original output size, and scaling factor α/r controls the magnitude of the update. The model learns to add a correction on top of the pretrained behavior rather than overwriting it.

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

Comment: SO instead of updating all weights of our pretrained model, LoRA freezes original weight matrix W and adds a small side path so made of two smaller matrices (linear layers) A and B that we initialize randomly and are trained. Their produc BA approximates wight update, but r is small so we train few parameters compared to full matrix. 

In forward pass, output is the original frozen path W(x) plus low rank updaye (α/r) * B(A(x)) and model learns to add correction on top of the pretrained behaviour rather than overwriting. We initialize B to zero as correction starts from zero so we start from pretrained state. 

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [31]:
r = 8
alpha = 16

lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Extract q_proj (query so input to q) and v_proj (value projection so content) from all attention layers, wrap in LoRA, put back
lora_targets = extract_lora_targets(lora_model) #get all attention layers' q_proj and v_proj by name, which are the target layers for LoRA
lora_layers = {name: LoRALayer(layer, r=r, alpha=alpha) for name, layer in lora_targets.items()} #for each target layer, create a LoRALayer that wraps the original layer and has the same input/output dimensions, with r=8 and alpha=16
lora_model = replace_layers(lora_model, lora_layers) #replace original q_proj and v_proj layers in the model with the new LoRALayer instances that wrap them, so now the model has the same architecture but with LoRA applied to those specific layers

print(f"Trainable:  {num_trainable_parameters(lora_model):,}")
print(f"Total:      {sum(p.numel() for p in lora_model.parameters()):,}")

lora_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=False, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

lora_trainer = make_trainer(lora_model, lora_training_args)
lora_trainer.train()
lora_eval_metrics = lora_trainer.evaluate()
print(lora_eval_metrics)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4989.72it/s]


Trainable:  121,704,768
Total:      134,975,808


  0%|          | 0/5000 [00:00<?, ?it/s]/Users/natalija.glisovic/Documents/phd_course_work/Deep Learning for NLP/labs/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 40%|████      | 2000/5000 [13:19<20:46,  2.41it/s]

{'loss': '1.648', 'grad_norm': '5.598', 'learning_rate': '3.001e-05', 'epoch': '0.4'}


 80%|████████  | 4000/5000 [26:18<06:47,  2.45it/s]

{'loss': '1.415', 'grad_norm': '8.926', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


100%|██████████| 5000/5000 [34:27<00:00,  2.42it/s]


{'eval_loss': '1.432', 'eval_rougeL': '0.6331', 'eval_runtime': '94.61', 'eval_samples_per_second': '4.228', 'eval_steps_per_second': '4.228', 'epoch': '1'}
{'train_runtime': '2067', 'train_samples_per_second': '2.419', 'train_steps_per_second': '2.419', 'train_loss': '1.493', 'epoch': '1'}


100%|██████████| 400/400 [01:33<00:00,  4.29it/s]

{'eval_loss': 1.432010531425476, 'eval_rougeL': 0.6330544201789149, 'eval_runtime': 93.416, 'eval_samples_per_second': 4.282, 'eval_steps_per_second': 4.282, 'epoch': 1.0}


Comment: I would expect fewer trainable parameters as we have r = 8 so fewer paramater to train, so also faster speed of training and I am thinking we might have lower ROUGE-L as we now do not finetune. I can see that this model ran slower (why? maybe because I ran on CPU so small extra operations might take longer). The evaluuation loss is smaller than both pre-train and SFT model as excepted and ROUGE-L larger as expected. 

lora 
'eval_loss': 1.432010531425476, 'eval_rougeL': 0.6330544201789149, 'eval_runtime': 93.416, 'eval_samples_per_second': 4.282, 'eval_steps_per_second': 4.282, 'epoch': 1.0
pretrain
PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.6193437576293945,
  "eval_model_preparation_time": 0.0023,
  "eval_rougeL": 0.5741986905444035,
  "eval_runtime": 65.2118,
  "eval_samples_per_second": 6.134,
  "eval_steps_per_second": 6.134,
  "epoch": 0
}

sft
'eval_loss': 1.5205596685409546, 'eval_rougeL': 0.6301315435201437, 'eval_runtime': 86.4252, 'eval_samples_per_second': 4.628, 'eval_steps_per_second': 4.628, 'epoch': 1.0}


### 🎓&nbsp; Task 4.4: Qualitative inspection (exam)

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [ ]:
def generate_response(model, user_input, system_prompt=None, max_new_tokens=100):
    prompt = ""
    if system_prompt:
        prompt += f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    prompt += f"<|im_start|>user\n{user_input}<|im_end|>\n<|im_start|>assistant\n"

    model.to("cpu")
    inputs = tokenizer(prompt, return_tensors="pt")  # already cpu by default
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:] #strip off the prompt tokens to get only the generated response tokens
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
lora_model.to("cpu").to(torch.float32) # do otherwise dtype mismatch error during generation since LoRALayer parameters are in float32 while original model might be in bf16 or fp16, so we convert the whole model to float32 on CPU for generation.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LoRALayer(
            (W): Linear(in_features=576, out_features=576, bias=False)
            (A): Linear(in_features=576, out_features=8, bias=False)
            (B): Linear(in_features=8, out_features=576, bias=False)
          )
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): LoRALayer(
            (W): Linear(in_features=576, out_features=192, bias=False)
            (A): Linear(in_features=576, out_features=8, bias=False)
            (B): Linear(in_features=8, out_features=192, bias=False)
          )
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_fe

In [44]:
user_input = "What is the capital of France?"

print("PRETRAINED:")
print(generate_response(pretrained_model, user_input))

print("\nSFT:")
print(generate_response(base_model, user_input))

print("\nLoRA:")
print(generate_response(lora_model, user_input))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


PRETRAINED:


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital of France?
What is the capital

SFT:


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


The capital of France is Paris.
##### Activity
What is the capital of France?
##### Activity
What is the capital of France?
##### Activity
What is the capital of France?
##### Activity
What is the capital of France?
##### Activity
What is the capital of France?
##### Activity
What is the capital of France?
##### Activity
What is the capital of France?


LoRA:
The capital of France is Paris.
#### 2020, 10:00 AM
##### 10:00 AM
##### 10:00 AM
##### 10:00 AM
##### 10:00 AM
##### 10:00 AM
##### 10:00 AM
##### 10:00 AM
##### 10:


Comment: Pretrained shows no instruction following at all (as expected as it just learned to continue text pattern). Both SFT and LoRA have learned instruction following to some extend as they correctly answer the same thing, but they suffer from repeyition after correct answers. SFT loops question back and LoRA loops timestamps, I think it's caus it hasn't learned a strong stopping point. 

In [45]:
user_input = "What is 1+1?"

print("PRETRAINED:")
print(generate_response(pretrained_model, user_input))

print("\nSFT:")
print(generate_response(base_model, user_input))

print("\nLoRA:")
print(generate_response(lora_model, user_input))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


PRETRAINED:


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


What is 1+1?
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+1=2
1+

SFT:


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


1 is the sum of two numbers.
1+1=2.
The sum of two numbers is 2.
The sum of 1 and 1 is 2.
The sum of 1 and 2 is 3.
The sum of 1 and 3 is 4.
The sum of 1 and 4 is 5.
The sum of 1 and 5 is 6.
The sum of 1 and 6 is

LoRA:
1+1 is the sum of two numbers.
The sum of two numbers is 1+1=2.
So, 1+1 is the sum of two numbers.
#### 1

#### 2

#### 3

#### 4

#### 5

#### 6

#### 7

#### 8

#### 9

#### 10

#### 11

#### 12


Comment: Same pattenr as above example, here both SFT and LoRA provide multiple responses that are all correct and interesting SFT gives more and more examples of summing two digits whereas w see LoRA has some issues in the end and just counts. 